<a href="https://colab.research.google.com/github/dorothea-pudding/114-1_TAICA_homework/blob/main/7-2.%20%E5%BB%BA%E7%AB%8B%E8%B3%87%E6%96%99%E6%9F%A5%E8%A9%A2%E6%A9%9F%E5%99%A8%E4%BA%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. 讀入你打造好的 vector dataset

In [ ]:
!pip -q install gdown

In [ ]:
GDRIVE_PUBLIC_URL = "https://drive.google.com/file/d/1oOYwsjdT1_5vqPSaY5W-MizJb7zJcgDz/view?usp=drive_link"

In [ ]:
!gdown --fuzzy -O faiss_db.zip "{GDRIVE_PUBLIC_URL}"

In [ ]:
!unzip faiss_db.zip

### 1. 安裝並引入必要套件

In [ ]:
#老師demo原版
"""
!pip install -U langchain langchain-community faiss-cpu transformers sentence-transformers huggingface_hub

!pip -q install "aisuite[all]" #安裝 AI Suite 套件及其全部選項（依賴套件很多，例如自動化工具、介面等）-q 表示安靜模式，不輸出過多訊息

!pip install -U langchain #單獨更新 langchain 到最新版本
!pip install -U faiss-cpu transformers sentence-transformers
!pip install -U langchain faiss-cpu transformers sentence-transformers huggingface_hub

!pip install -U langchain langchain-community==0.0.29 faiss-cpu sentence-transformers huggingface_hub
!pip install -U langchain-openai==0.0.5 #安裝 langchain-openai 版本 0.0.5 這個套件提供與 OpenAI API 互動的模組

!pip install pydantic==1.10.14
"""

In [ ]:
#ChatGPT版
"""
# 核心 LLM 套件
!pip install langchain==0.1.14 \
        langchain-community==0.0.30 \
        langchain-openai==0.0.5 \
        faiss-cpu \
        transformers \
        sentence-transformers \
        huggingface_hub

# AI Suite
!pip install "aisuite[all]"

# OpenAI、Gradio
!pip install openai gradio

# 確保 numpy 和 pydantic 版本穩定
!pip install pydantic==1.10.14
"""

In [ ]:
#助教版
!pip install -U langchain langchain-community faiss-cpu transformers sentence-transformers huggingface_hub
!pip install -U langchain-huggingface
!pip install -U langchain-openai


In [ ]:
!pip install gradio

In [ ]:
##import部分，老師demo版加ChatGPTz03ChatGPT反覆修正，無法執行原因待檢查
"""
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain

from openai import OpenAI
import gradio as gr
"""

In [ ]:
#助教版
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

In [ ]:
from openai import OpenAI
import gradio as gr

### 2. 自訂 E5 embedding 類別

In [ ]:
import os
from google.colab import userdata

In [ ]:
hf_token = userdata.get('Hugging_face')

In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
class EmbeddingGemmaEmbeddings(HuggingFaceEmbeddings):
    def __init__(self, **kwargs):
        super().__init__(
            #model_name="google/embeddinggemma-300m",
            encode_kwargs={"normalize_embeddings": True},
            **kwargs
        )

    def embed_documents(self, texts):
        # 你也可以把 "none" 改成真實標題（檔名/章節名），效果會更穩
        texts = [f"title: none | text: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        # 官方檢索建議前綴
        return super().embed_query(f"task: search result | query: {text}")

### 3. 載入 `faiss_db`

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    "faiss_db",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### 4. 設定好我們要的 LLM

如之前, 我們會用 OpenAI API。這裡使用 Groq 服務, 可改成你要的服務。

In [ ]:
!pip -q install -U groq aisuite

In [ ]:
import aisuite as ai

In [ ]:
from google.colab import userdata

api_key = userdata.get('Groq')

In [ ]:
os.environ['GROQ_API_KEY']=api_key

這裡的模型和 `base_url` 是用 Groq, 如果用其他服務請自行修改。

In [ ]:
model = "groq:openai/gpt-oss-120b"
#base_url="https://api.groq.com/openai/v1"

In [ ]:
client = ai.Client()

### 5. `prompt` 設計

In [ ]:
system_prompt = """你是龍華科大的的 AI 修課輔導人員，請根據資料來回應學生的問題。
請親切、簡潔（可條列）並附帶具體建議。
請用台灣習慣的中文回應。"""

prompt_template = """
根據下列資料：
{retrieved_chunks}

回答使用者的問題：{question}

請根據資料內容回覆，若資料不足請告訴同學可以請教老師或相關校務人員。
"""

### 6. 使用 RAG 來回應

搜尋與使用者問題相關的資訊，根據我們的 prompt 樣版去讓 LLM 回應。

In [ ]:
#老師demo版
"""
chat_history = []

def chat_with_rag(user_input):
    global chat_history
    # 取回相關資料
    docs = retriever.get_relevant_documents(user_input)
    retrieved_chunks = "\n\n".join([doc.page_content for doc in docs])

    # 將自定 prompt 套入格式
    final_prompt = prompt_template.format(retrieved_chunks=retrieved_chunks, question=user_input)

    # 用 AI Suite 呼叫語言模型
    response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": final_prompt},
    ]
    )
    answer = response.choices[0].message.content

    chat_history.append((user_input, answer))
    return answer
    """

In [ ]:
##ChatGPT版
"""
rag_chain = ConversationalRetrievalChain.from_llm(llm, retriever=retriever)

def chat_with_rag(user_input, chat_history=None):
    if chat_history is None:
        chat_history = []

    # 使用已經建立的 rag_chain
    result = rag_chain({"question": user_input, "chat_history": chat_history})
    answer = result["answer"]

    # 更新對話歷史
    chat_history.append((user_input, answer))
    return answer, chat_history
    """

In [ ]:
#因為ChatGPT開始鬼打牆，所以我選擇把這段和使用gradio的部分交給Gemini接力
# -------------------------------------------------------------
# 儲存格一：環境設定與 RAG 核心函數
# -------------------------------------------------------------

# 請確保在此處或之前的儲存格中，您已安裝必要的庫（如 pip install gradio）
# 並定義和初始化所有 RAG 相關的物件：
# 例如：
# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import GoogleGenAIEmbeddings
# from google import genai
# import gradio as gr

# # 假設這是您初始化 RAG 組件的部分
# # (請替換為您的實際初始化代碼)
# # -------------------------------------------------------------
# # client = genai.Client(api_key="YOUR_API_KEY")
# # model = "gemini-2.5-flash"
# # system_prompt = "你是一個畢業條件輔導員，請根據提供的資料回答學生的問題。"
# # prompt_template = """請根據以下檢索到的資料來回答問題：

# # --- 檢索資料 ---
# # {retrieved_chunks}
# # --- 檢索資料 ---

# # 這是學生的問題：{question}
# # 你的回答："""
# # # 假設您已經有了一個名為 retriever 的有效物件
# # # retriever = ...
# # # -------------------------------------------------------------


def chat_with_rag(user_input):
    """
    RAG 核心查詢函數。接收用戶輸入，執行檢索，生成最終 Prompt，調用語言模型並返回答案。
    (此處使用修正後的邏輯，不處理 Gradio 的 global 歷史)
    """

    # 1. 取回相關資料 (假設 retriever 變數已定義)
    try:

        # **【最終修正點】** # 針對最新的 LangChain 版本，使用 .invoke() 方法
        # .invoke() 接收查詢字串並返回 List[Document]
        docs = retriever.invoke(user_input)

        # 確保返回的是 List[Document]，如果是則可以繼續
        retrieved_chunks = "\n\n".join([doc.page_content for doc in docs])

    except NameError:
        return "錯誤：retriever 物件未定義。"
    except NameError:
        return "錯誤：retriever 物件未定義或無法調用 get_relevant_documents。"
    except Exception as e:
        return f"錯誤：調取相關資料時發生問題：{e}"


    # 2. 將自定 prompt 套入格式 (假設 prompt_template 變數已定義)
    try:
        final_prompt = prompt_template.format(retrieved_chunks=retrieved_chunks, question=user_input)
    except NameError:
       return "錯誤：prompt_template 未定義。"


    # 3. 用 AI Suite 呼叫語言模型 (假設 client, model, system_prompt 變數已定義)
    try:
        # 使用您原有的 API 呼叫格式
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": final_prompt},
            ]
        )
        answer = response.choices[0].message.content
    except NameError:
        return "錯誤：client, model 或 system_prompt 未定義。"
    except Exception as e:
        # 捕獲 API 呼叫或連線相關錯誤
        return f"錯誤：呼叫語言模型時發生問題：{e}"

    # 4. 返回答案
    return answer

print("RAG 核心函數定義完成。請執行下一個儲存格以啟動 Gradio 介面。")

### 7. 用 Gradio 打造 Web App

In [ ]:
#老師demo版
"""
with gr.Blocks() as demo:
    gr.Markdown("# 🎓 AI 修課／畢業條件輔導員")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="請輸入你的問題...")

    def respond(message, chat_history_local):
        response = chat_with_rag(message)
        chat_history_local.append((message, response))
        return "", chat_history_local

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch(debug=True)
"""

In [ ]:
##ChatGPT 版
"""
with gr.Blocks() as demo:
    gr.Markdown("# 🎓 AI 修課／畢業條件輔導員")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="請輸入你的問題...")

    def respond(message, chat_history_local):
        response, chat_history_local = chat_with_rag(message, chat_history_local)
        return "", chat_history_local

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch(debug=True)
"""

In [ ]:
# -------------------------------------------------------------
# 儲存格二：Gradio 介面與啟動
# -------------------------------------------------------------

import gradio as gr # 確保 Gradio 已匯入

# Gradio 的回應函數，處理歷史記錄
def respond(message, chat_history_list):
    """
    接收用戶訊息和當前歷史記錄，呼叫 RAG 函數，並更新歷史記錄列表。
    """
    # 1. 取得 RAG 回覆
    response = chat_with_rag(message)

    # 2. 更新聊天記錄列表 (Gradio Chatbot 的標準格式：[[user, bot], ...])
    chat_history_list.append((message, response))

    # 3. 返回新的空輸入框內容和更新後的聊天記錄列表
    return "", chat_history_list


with gr.Blocks() as demo:
    gr.Markdown("# 🎓 AI 修課／畢業條件輔導員")

    # chatbot 組件，用於顯示聊天記錄
    # value=[] 確保初始值為空列表
    chatbot = gr.Chatbot(value=[], label="聊天記錄")

    # 文本框組件，用於用戶輸入
    msg = gr.Textbox(placeholder="請輸入你的問題...")

    # 清除按鈕
    clear_btn = gr.Button("清除聊天記錄")


    # 定義事件：當用戶提交輸入時 (按 Enter 或點擊隱藏的提交按鈕)
    # 輸入：[msg 的值, chatbot 的值 (歷史列表)]
    # 輸出：[更新 msg (清空), 更新 chatbot (新的歷史列表)]
    msg.submit(respond, [msg, chatbot], [msg, chatbot], queue=False)

    # 定義事件：點擊清除按鈕時
    # 輸出：清空 msg (返回 None), 清空 chatbot (返回 [])
    clear_btn.click(lambda: (None, []), None, [msg, chatbot], queue=False)


# 啟動 Gradio 應用
# 在 Colab 中，launch() 會顯示一個公共 URL 或內嵌介面
demo.launch(debug=True, share=True)

In [ ]:
"""
附註（一些遇到的主要問題）：
1.老師demo中用的套件版本似乎已經過期，所以需要另外查資料跟讓AI幫忙查詢可行方案
2.在一些比較長的程式（比如最後用gradio的部分）當中，ChatGPT容易鬼打牆，但是在釐清概念和修正短程式方面沒有問題
  但是在需要長一點的程式重寫就需要Gemini幫忙
3.問題比較多是出在LangChain函式庫的更新，所以多花很多時間在找資料、與AI討論、釐清概念和問題
3.遇到的主要問題（解釋來源於Gemini）:
這個問題的出現，主要是由於您所使用的 RAG 框架（幾乎可以確定是 LangChain 庫）在不同版本之間，特別是在 Retriever 介面上的設計變更和演進所造成的。
以下是詳細的解釋：
1. 錯誤的根源：BaseRetriever 介面的變化
您遇到的錯誤是：
'VectorStoreRetriever' object has no attribute 'get_relevant_documents'
這意味著 retriever 這個物件在 Python 執行時被認定為 VectorStoreRetriever 類型的實例，但它不包含名為 get_relevant_documents 的方法。
舊版本 LangChain (<= 0.1.x)：
在 LangChain 較舊的版本中，BaseRetriever 類別定義了 get_relevant_documents(query: str) 作為檢索文檔的主要同步方法。
 * 如果您當時使用這個版本，您的原始碼是正確的。
新版本 LangChain (LangChain 0.2.x 或更高，強調 LCEL)：
LangChain 為了建立更一致、更高效能的 RAG 鏈，引入了 LangChain Expression Language (LCEL) 和 Runnable 介面。
在新的設計中：
 * 標準同步執行方法被替換為 invoke()：
   * BaseRetriever 及其子類（如 VectorStoreRetriever）繼承了 Runnable 的特性。
   * 對於所有 Runnable 類型的物件，標準的同步執行入口點是 .invoke(input)。它接收一個輸入（在您的案例中是查詢字串）並返回輸出（相關的 Document 列表）。
 * get_relevant_documents 變成了內部或非同步方法：
   * 在某些新版本中，get_relevant_documents 可能被移除了，或被標記為僅供內部使用，或被非同步方法 aget_relevant_documents 所取代。當您嘗試在新的 VectorStoreRetriever 實例上呼叫它時，Python 會因為找不到這個屬性而拋出 AttributeError。
2. 解決方案的原理：擁抱 Runnable 介面
您通過使用 .invoke(user_input) 解決了問題。
原理：
當您執行 retriever.invoke(user_input) 時：
 * Python 發現 VectorStoreRetriever 擁有 invoke 方法（因為它繼承了 Runnable）。
 * invoke 方法在內部執行 VectorStoreRetriever 的核心邏輯，即根據 user_input 執行向量相似度搜尋。
 * 它成功返回了所需的 List[Document]，滿足了您 RAG 程式碼中 docs 變數的需求。
總結來說，您的問題是一個典型的庫版本升級導致的 API 不相容問題。程式碼邏輯本身是正確的，但您使用的 LangChain 版本要求您使用新的標準方法 .invoke()，而不是舊版本常用的 .get_relevant_documents()。
因此，修正的本質就是將程式碼從舊的 API 呼叫（retriever.get_relevant_documents）遷移到了新的標準 API 呼叫（retriever.invoke）。
